# GCN Example 4 (PyTorch) — Graph Attention Network (GAT)

PyTorch version of `gat_karate_club.ipynb`. Same Karate Club node-classification task,
but neighbours are aggregated with **learned attention weights** instead of the fixed
degree-based weights a GCN uses.

For node $i$ and neighbour $j$: score $e_{ij} = \text{LeakyReLU}(a^\top[Wh_i \| Wh_j])$,
softmax over neighbours to get $\alpha_{ij}$, then $h_i^{new} = \sigma(\sum_j \alpha_{ij} W h_j)$.
We compute the dense attention matrix and mask non-edges before the softmax.

> Self-contained, runs instantly on CPU. Needs `torch`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Graph, features, labels

78 edges, identity features, only nodes 0 and 33 labelled. The adjacency includes
self-loops and becomes a boolean `mask` for attention.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]
N = 34
A = np.zeros((N, N), dtype='float32')
for i, j in edges:
    A[i, j] = A[j, i] = 1.0
A_self = A + np.eye(N, dtype='float32')
mask = torch.tensor(A_self, device=device) > 0

X = torch.eye(N, device=device)
y_true = np.array([0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,
                   1,1,1,1,1,1,1,1,1,1])
y_true_t = torch.tensor(y_true, dtype=torch.long, device=device)
labeled_nodes = torch.tensor([0, 33], device=device)
y_labeled = y_true_t[labeled_nodes]
print('Edges:', len(edges), '| labelled:', labeled_nodes.tolist())

## 2. A Graph Attention layer

Dense single-head GAT. We use `a^T[Wh_i || Wh_j] = a_src·Wh_i + a_dst·Wh_j` to build the
`(N,N)` score matrix from two `(N,1)` columns, mask non-edges with `-inf`, softmax, and
aggregate.

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a_src = nn.Linear(out_features, 1, bias=False)
        self.a_dst = nn.Linear(out_features, 1, bias=False)
        self.alpha = None

    def forward(self, x, mask):
        h = self.W(x)                                  # (N, out)
        scores = self.a_src(h) + self.a_dst(h).T       # (N, N) via broadcasting
        scores = F.leaky_relu(scores, 0.2)
        scores = scores.masked_fill(~mask, float('-inf'))
        alpha = F.softmax(scores, dim=1)
        self.alpha = alpha.detach()
        return alpha @ h                               # weighted aggregation

class GAT(nn.Module):
    def __init__(self, in_features, hidden=8, classes=2):
        super().__init__()
        self.l1 = GATLayer(in_features, hidden)
        self.l2 = GATLayer(hidden, classes)

    def forward(self, x, mask):
        h = F.elu(self.l1(x, mask))
        return self.l2(h, mask)

model = GAT(N).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 3. Train

In [ ]:
EPOCHS = 300
for epoch in range(1, EPOCHS + 1):
    model.train()
    logits = model(X, mask)
    loss = F.cross_entropy(logits[labeled_nodes], y_labeled)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 30 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            acc = (model(X, mask).argmax(1) == y_true_t).float().mean().item()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss.item():.4f}  full-graph acc={acc:.3f}')

## 4. Results and attention visualization

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X, mask).argmax(1).cpu().numpy()
print(f'Final accuracy on all {N} nodes: {(preds == y_true).mean():.1%}')
print('Predicted:', preds)
print('True:     ', y_true)

alpha = model.l1.alpha.cpu().numpy()
plt.figure(figsize=(6, 5))
plt.imshow(alpha, cmap='viridis')
plt.colorbar(label='attention weight')
plt.xlabel('neighbour j'); plt.ylabel('node i')
plt.title('Layer-1 attention weights (row-normalized)')
plt.show()

### Things to try

- Add **multiple heads** (several `GATLayer`s in parallel, concatenate outputs).
- Apply dropout to the attention weights.
- Compare accuracy and embeddings against the plain GCN.
- See which neighbours nodes 0 and 33 attend to most.